# Notebook 1 — Exploratory Data Analysis & Visualization
## Ames Housing Dataset (2,930 sales, 82 columns)

**Purpose**: Explore the raw dataset to understand distributions, missing values,
correlations, and outliers. These findings guide cleaning and feature selection
decisions in Notebooks 2 and 3.

**No fitting occurs here** — the full dataset is used for exploration only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")
pd.set_option("display.max_columns", 85)

RANDOM_STATE = 42

## 1) Data Loading and Initial Inspection

The Ames Housing dataset (ISU version) has **2,930 residential sales** in Ames, Iowa
with **82 columns** (80 features + Order + PID identifiers). Column names contain
spaces (e.g., "Overall Qual") — we strip them to CamelCase for consistency with the
Kaggle convention and downstream code.

In [ ]:
# Load from local CSV (ISU version — 2,930 rows vs Kaggle's 1,460)
df = pd.read_csv("../data/raw/AmesHousing.csv")

# Normalize column names: strip spaces and slashes -> CamelCase
df.columns = df.columns.str.replace(" ", "").str.replace("/", "")

# Drop database identifiers (no predictive value)
df = df.drop(columns=["Order", "PID"])

print(f"Shape: {df.shape}")
print(f"\nColumn type breakdown:")
print(df.dtypes.value_counts())
print(f"\nNumeric columns: {df.select_dtypes(include='number').shape[1]}")
print(f"Categorical columns: {df.select_dtypes(include='object').shape[1]}")
df.head()

In [ ]:
# Summary statistics for numeric columns
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

In [ ]:
# Categorical column overview: cardinality
cat_cols = df.select_dtypes(include="object").columns
cat_cardinality = df[cat_cols].nunique().sort_values(ascending=False)
print(f"Categorical columns ({len(cat_cols)}):\n")
print(cat_cardinality.to_string())

In [ ]:
# Sample values from key columns to understand the data
sample_cols = ["OverallQual", "BsmtQual", "Neighborhood", "KitchenQual"]
for col in sample_cols:
    print(f"{col}: {df[col].unique()[:10].tolist()}")
    print()

## 2) Target Variable Analysis — SalePrice

Before any modeling, we need to understand the distribution of our target. A
right-skewed target can hurt linear models — log-transformation is the standard fix.

In [ ]:
# SalePrice summary stats
print("SalePrice summary:")
print(df["SalePrice"].describe().round(0))
print(f"\nSkewness: {df['SalePrice'].skew():.3f}")
print(f"Kurtosis: {df['SalePrice'].kurtosis():.3f}")

# Two-panel figure: histogram + QQ plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram with KDE, mean/median lines
sns.histplot(df["SalePrice"], kde=True, bins=50, ax=axes[0])
axes[0].axvline(df["SalePrice"].mean(), color="red", linestyle="--", label=f"Mean: ${df['SalePrice'].mean():,.0f}")
axes[0].axvline(df["SalePrice"].median(), color="green", linestyle="--", label=f"Median: ${df['SalePrice'].median():,.0f}")
axes[0].set_title("SalePrice Distribution")
axes[0].legend()

# QQ plot
stats.probplot(df["SalePrice"], plot=axes[1])
axes[1].set_title("SalePrice QQ Plot")

plt.tight_layout()
plt.show()

In [ ]:
# Log-transform preview — does it fix the skew?
log_price = np.log1p(df["SalePrice"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(log_price, kde=True, bins=50, ax=axes[0])
axes[0].set_title(f"log1p(SalePrice) — Skewness: {log_price.skew():.3f}")
axes[0].set_xlabel("log1p(SalePrice)")

stats.probplot(log_price, plot=axes[1])
axes[1].set_title("log1p(SalePrice) QQ Plot")

plt.tight_layout()
plt.show()

print(f"Raw skewness:  {df['SalePrice'].skew():.3f}")
print(f"Log skewness:  {log_price.skew():.3f}")
print("→ Log-transform reduces skewness close to 0 — will use for training.")

In [ ]:
# SalePrice quartile breakdown
quartiles = pd.qcut(df["SalePrice"], q=4, labels=["Q1 (cheap)", "Q2", "Q3", "Q4 (expensive)"])
print("SalePrice quartile breakdown:")
print(quartiles.value_counts().sort_index())
print(f"\nQuartile boundaries: {pd.qcut(df['SalePrice'], q=4, retbins=True)[1].round(0).tolist()}")

## 3) Missing Value Audit

Always audit missing values **before** splitting or fitting anything. We need to
understand *why* values are missing — structural absence (e.g., no garage → NaN) vs.
recording gaps (e.g., LotFrontage not measured).

In [ ]:
# Null summary: count, percentage, dtype
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct,
    "dtype": df.dtypes
}).query("null_count > 0").sort_values("null_pct", ascending=False)

print(f"Columns with nulls: {len(null_summary)} out of {df.shape[1]}")
print(f"Total null values: {df.isnull().sum().sum():,}\n")
null_summary

In [ ]:
# Missing value heatmap — shows structural co-occurrence patterns
plt.figure(figsize=(16, 8))
sns.heatmap(df[null_summary.index].isnull(), cbar=True, yticklabels=False, cmap="viridis")
plt.title("Missing Value Pattern Heatmap (columns with nulls only)")
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal bar chart of null percentages with threshold lines
plt.figure(figsize=(12, 6))
null_summary["null_pct"].sort_values(ascending=True).plot(
    kind="barh", color=["red" if x > 40 else "orange" if x > 5 else "green"
                        for x in null_summary["null_pct"].sort_values(ascending=True)]
)
plt.axvline(x=40, color="red", linestyle="--", alpha=0.7, label="40% drop threshold")
plt.axvline(x=5, color="orange", linestyle="--", alpha=0.7, label="5% structural boundary")
plt.xlabel("Null Percentage (%)")
plt.title("Missing Values by Column")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Null co-occurrence: garage columns are null on the same rows
garage_cols = ["GarageType", "GarageFinish", "GarageQual", "GarageCond", "GarageYrBlt"]
garage_nulls = df[garage_cols].isnull()
garage_null_rows = df[garage_nulls.any(axis=1)]

print(f"Rows with any garage null: {len(garage_null_rows)}")
print(f"When GarageType is NaN, GarageArea and GarageCars are:")
print(df.loc[df["GarageType"].isnull(), ["GarageArea", "GarageCars"]].describe())

# Basement columns — same pattern
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
bsmt_nulls = df[bsmt_cols].isnull()
bsmt_null_rows = df[bsmt_nulls.any(axis=1)]

print(f"\nRows with any basement null: {len(bsmt_null_rows)}")
print(f"When BsmtQual is NaN, TotalBsmtSF is:")
print(df.loc[df["BsmtQual"].isnull(), "TotalBsmtSF"].describe())
print("\n→ Confirms: these nulls are STRUCTURAL (no garage/basement), not recording errors.")

## 4) Feature Type Classification

Distinguishing numeric, ordinal (ordered categories), and nominal (unordered categories)
is essential for choosing the right encoding and statistical tests later.

In [ ]:
# Feature type classification
# Numeric: continuous or discrete numeric values
numeric_features = df.select_dtypes(include="number").drop(columns=["SalePrice"]).columns.tolist()

# Ordinal: categorical with a natural ordering (quality/condition scales)
ordinal_features = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
    "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "GarageFinish",
    "Functional", "LandSlope", "LotShape", "PavedDrive", "Fence",
]

# Nominal: categorical with no natural ordering
nominal_features = [c for c in df.select_dtypes(include="object").columns
                    if c not in ordinal_features]

print(f"Numeric features:  {len(numeric_features)}")
print(f"Ordinal features:  {len(ordinal_features)}")
print(f"Nominal features:  {len(nominal_features)}")
print(f"Total (excl. target): {len(numeric_features) + len(ordinal_features) + len(nominal_features)}")
print(f"\nOrdinal: {ordinal_features}")
print(f"\nNominal: {nominal_features}")

## 5) Correlation Analysis

Pearson correlation measures linear relationships between numeric features and SalePrice.
Features with |r| > 0.5 are strong candidates for the model.

In [ ]:
# Full Pearson correlation heatmap (numeric features + target)
corr_cols = numeric_features + ["SalePrice"]
corr_matrix = df[corr_cols].corr()

# Mask upper triangle for cleaner visual
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Pearson Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 features by absolute Pearson correlation with SalePrice
corr_with_target = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
corr_with_target.head(15).sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("|Pearson r|")
plt.title("Top 15 Features by Correlation with SalePrice")
plt.tight_layout()
plt.show()

print("Top 15 correlations with SalePrice:")
for feat, r in corr_with_target.head(15).items():
    print(f"  {feat:20s}  |r| = {r:.3f}")

## 6) Scatter Plots: Top Features vs SalePrice

Visual confirmation of linear relationships. Strong correlations should show
a clear upward trend with tight scatter.

In [ ]:
# 2x3 scatter grid for top 6 numeric features
top6 = corr_with_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for idx, feat in enumerate(top6):
    ax = axes[idx // 3, idx % 3]
    ax.scatter(df[feat], df["SalePrice"], alpha=0.2, s=10)

    # Regression line
    mask = df[feat].notna() & df["SalePrice"].notna()
    if mask.sum() > 2:
        z = np.polyfit(df.loc[mask, feat], df.loc[mask, "SalePrice"], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(df.loc[mask, feat])
        ax.plot(x_sorted, p(x_sorted), color="red", linewidth=2)

    r = corr_matrix.loc[feat, "SalePrice"]
    ax.set_title(f"{feat} (r = {r:.3f})")
    ax.set_ylabel("SalePrice")

plt.suptitle("Top 6 Numeric Features vs SalePrice", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7) Categorical Feature Analysis

Box plots show how SalePrice varies across categorical groups. High between-group
variance indicates strong predictive power.

In [ ]:
# 2x2 box plots for key categorical features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Neighborhood sorted by median price
neighborhood_order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=neighborhood_order, ax=axes[0, 0])
axes[0, 0].set_title("Neighborhood vs SalePrice (sorted by median)")
axes[0, 0].tick_params(axis="x", rotation=90)

# OverallQual
sns.boxplot(data=df, x="OverallQual", y="SalePrice", ax=axes[0, 1])
axes[0, 1].set_title("OverallQual vs SalePrice")

# KitchenQual (ordered)
kitchen_order = ["Po", "Fa", "TA", "Gd", "Ex"]
sns.boxplot(data=df, x="KitchenQual", y="SalePrice", order=kitchen_order, ax=axes[1, 0])
axes[1, 0].set_title("KitchenQual vs SalePrice (ordered)")

# GarageType
sns.boxplot(data=df.dropna(subset=["GarageType"]), x="GarageType", y="SalePrice", ax=axes[1, 1])
axes[1, 1].set_title("GarageType vs SalePrice")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 8) Ordinal Feature Analysis

Ordinal features have a natural ranking (Po < Fa < TA < Gd < Ex). Violin plots show
whether SalePrice increases monotonically with quality — confirming the ordinal ordering
is meaningful for prediction.

In [ ]:
# Violin plots for key ordinal features
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]
ordinal_plot_cols = ["ExterQual", "BsmtQual", "HeatingQC", "KitchenQual"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, col in enumerate(ordinal_plot_cols):
    ax = axes[idx // 2, idx % 2]
    # Filter to existing quality values
    valid_order = [v for v in quality_order if v in df[col].dropna().unique()]
    data = df.dropna(subset=[col])
    sns.violinplot(data=data, x=col, y="SalePrice", order=valid_order, ax=ax)
    ax.set_title(f"{col} vs SalePrice")

plt.suptitle("Ordinal Features vs SalePrice (ordered low → high quality)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9) Outlier Detection

The Ames dataset has well-documented outliers — notably two properties with
GrLivArea > 4,000 sqft sold below $200k. These are likely non-arm's-length transactions
and should be removed from training data.

In [ ]:
# GrLivArea vs SalePrice — highlight known outliers
fig, ax = plt.subplots(figsize=(10, 6))

outlier_mask = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 200000)
normal = df[~outlier_mask]
outliers = df[outlier_mask]

ax.scatter(normal["GrLivArea"], normal["SalePrice"], alpha=0.3, s=15, label="Normal")
ax.scatter(outliers["GrLivArea"], outliers["SalePrice"], color="red", s=80,
           marker="X", label=f"Outliers ({len(outliers)})")
ax.set_xlabel("GrLivArea (sqft)")
ax.set_ylabel("SalePrice ($)")
ax.set_title("GrLivArea vs SalePrice — Known Outliers Highlighted")
ax.legend()
plt.tight_layout()
plt.show()

# IQR-based outlier counts for top 10 numeric features
print("\nIQR-based outlier counts (top 10 correlated features):")
for feat in corr_with_target.head(10).index:
    q1 = df[feat].quantile(0.25)
    q3 = df[feat].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[feat] < lower) | (df[feat] > upper)).sum()
    print(f"  {feat:20s}  outliers: {n_outliers:4d}  ({n_outliers/len(df)*100:.1f}%)")

## 10) Feature Engineering Preview

Combining related features can produce stronger predictors than individual components.
We test four engineered features and compare their correlations with SalePrice.

In [ ]:
# Create candidate engineered features
df_eng = df.copy()
df_eng["TotalSF"] = df_eng["TotalBsmtSF"].fillna(0) + df_eng["1stFlrSF"] + df_eng["2ndFlrSF"]
df_eng["HouseAge"] = df_eng["YrSold"] - df_eng["YearBuilt"]
df_eng["TotalBath"] = (df_eng["FullBath"] + 0.5 * df_eng["HalfBath"]
                       + df_eng["BsmtFullBath"].fillna(0) + 0.5 * df_eng["BsmtHalfBath"].fillna(0))
df_eng["HasRemodel"] = (df_eng["YearRemodAdd"] != df_eng["YearBuilt"]).astype(int)

# Compare engineered vs component correlations
engineered = {
    "TotalSF": ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"],
    "HouseAge": ["YearBuilt"],
    "TotalBath": ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"],
    "HasRemodel": ["YearRemodAdd"],
}

print("Engineered feature correlations with SalePrice:\n")
all_corrs = {}
for eng_feat, components in engineered.items():
    r_eng = df_eng[eng_feat].corr(df_eng["SalePrice"])
    all_corrs[eng_feat] = r_eng
    comp_rs = {c: df_eng[c].corr(df_eng["SalePrice"]) for c in components}
    print(f"  {eng_feat:15s}  r = {r_eng:+.3f}  (components: {', '.join(f'{c}={r:+.3f}' for c, r in comp_rs.items())})")
    for c, r in comp_rs.items():
        all_corrs[c] = r

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#d63031" if k in engineered else "#0984e3" for k in all_corrs]
pd.Series(all_corrs).sort_values().plot(kind="barh", color=colors, ax=ax)
ax.set_xlabel("Pearson r with SalePrice")
ax.set_title("Engineered Features (red) vs Components (blue)")
plt.tight_layout()
plt.show()

## 11) Key Findings

1. **SalePrice is right-skewed** (skewness ~1.7) → log-transform for training
2. **Top numeric predictors**: OverallQual, GrLivArea, GarageCars/GarageArea, TotalBsmtSF, 1stFlrSF, YearBuilt
3. **Key categorical**: Neighborhood (high between-group variance), KitchenQual/ExterQual (strong ordinal)
4. **Structural nulls dominate**: garage, basement, pool, fence → fill "None"/0 (feature absent)
5. **LotFrontage**: ~17% missing — genuine recording gap → median imputation
6. **Engineered features**: TotalSF and TotalBath show stronger correlations than individual components
7. **Two GrLivArea outliers** (>4000 sqft, <$200k) flagged for removal from training

**Next**: Notebook 2 applies these findings — split, clean, engineer, save processed data.